# Dynamic Validation Execution Engine

## Version 2 - Phase 2

In this notebook, we are building a dynamic metadata-driven validation engine.

Objectives:
- dynamically read validation rules
- automatically generate validation conditions
- dynamically create validation status columns
- reduce hardcoded validation implementations
- improve framework scalability

This notebook transforms the project into a reusable enterprise-grade validation platform.

#### Step 1: Import Required Libraries

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

#### Step 2: Load Bronze Dataset

We are loading the raw Bronze layer dataset for dynamic validation processing.

In [0]:
df = spark.read.parquet(
    "/Volumes/taxi_quality_platform/bronze/raw_taxi_data/yellow_tripdata_2024-01.parquet"
)

#### Step 3: Load Validation Metadata Rules

We are loading configurable validation rules from the metadata configuration table.

In [0]:
rules_df = spark.read.table(
    "taxi_quality_platform.config.validation_rules"
)

#### Step 4: Convert Rules DataFrame into Python Collection

We are converting metadata rules into Python objects for dynamic rule execution.

In [0]:
rules = rules_df.collect()

#### Step 5: Dynamically Apply Validation Rules

We are dynamically generating validation status columns based on metadata-driven rules.

In [0]:
validated_df = df

In [0]:
for rule in rules:

    column_name = rule["column_name"]
    validation_type = rule["validation_type"]
    rule_value = rule["rule_value"]
    validation_column_name = f"{column_name}_validation_status"

    if validation_type == "greater_than":
        validated_df = validated_df.withColumn(
            validation_column_name,
            when(col(column_name) > float(rule_value), "PASSED")
            .otherwise("FAILED")
        )

    elif validation_type == "between":
        min_value = rule_value.split(",")[0]
        max_value = rule_value.split(",")[1]
        
        validated_df = validated_df.withColumn(
            validation_column_name,
            when(col(column_name).between(float(min_value), float(max_value)), "PASSED")
            .otherwise("FAILED")
        )

    elif validation_type == "not_null":
        validated_df = validated_df.withColumn(
             validation_column_name,
             when(col(column_name).isNotNull(), "PASSED")
             .otherwise("FAILED")
        )

#### Step 6: Display Dynamically Generated Validation Columns

In [0]:
validated_df.show(5)

#### Step 7: Identify Validation Status Columns

We are dynamically identifying all generated validation status columns.

In [0]:
validation_columns = [
    column
    for column in validated_df.columns
    if column.endswith("_validation_status")
]

In [0]:
print(validation_columns)

#### Step 8: Generate Overall Validation Status Dynamically

A record is marked as FAILED if any validation fails.

In [0]:
validated_df = validated_df.withColumn(
    "overall_validation_status",

    when(
        array_contains(                                 # array_contains will return true if the array contains the value "FAILED"
            array(
                *[                                      # since array_contains will take input as array of values * will help to unpack the list items 
                    col(column)
                    for column in validation_columns    # this list compression will return the list of columns with _validation_status 
                ]
            ),
            "FAILED"
        ),
        "FAILED"
    ).otherwise("PASSED")
)


#### Step 9: Generate Passed and Failed Record Datasets

In [0]:
valid_records_df = validated_df.filter(
    col("overall_validation_status") == "PASSED"
)

invalid_records_df = validated_df.filter(
    col("overall_validation_status") == "FAILED"
)

In [0]:
print("Total Records:", validated_df.count())

print("Valid Records:", valid_records_df.count())

print("Invalid Records:", invalid_records_df.count())

#### Step 10: Generate Dynamic Validation Summary

We are generating validation statistics dynamically from metadata-driven validations.

In [0]:
validation_summary = []

for column in validation_columns:

    failed_count = validated_df.filter(col(column) == "FAILED").count()

    passed_count = validated_df.filter(col(column) == "PASSED").count()

    validation_summary.append(
        (
        column,
        passed_count,
        failed_count
        )
    )

In [0]:
summary_schema = StructType([
    StructField("column", StringType(), True),
    StructField("passed_count", LongType(), True),
    StructField("failed_count", LongType(), True)
])

In [0]:
validation_summary_df = spark.createDataFrame(
    validation_summary,
    summary_schema
)

In [0]:
validation_summary_df.show()

#### Step 11: Store Dynamic Validation Summary Table

We are storing dynamically generated validation statistics for monitoring and analytics.

In [0]:
validation_summary_df.write\
    .format("delta")\
        .mode("overwrite")\
            .saveAsTable( "taxi_quality_platform.gold.dynamic_validation_summary")

#### Dynamic Validation Engine Summary

In this notebook, we:
- dynamically loaded validation metadata rules
- automatically generated validation logic
- dynamically created validation status columns
- generated validation summaries
- reduced hardcoded validation implementations
- improved framework scalability

This notebook significantly upgraded the project into a reusable enterprise-grade validation platform.